In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from pathlib import Path

input_dirs = list(Path("/kaggle/input").iterdir())
print("Attached inputs:", input_dirs)

pngs = sorted(Path("/kaggle/input").rglob("*.png"))
print("PNG count:", len(pngs))
for p in pngs[:10]:
  print(p)

Attached inputs: [PosixPath('/kaggle/input/datasets')]
PNG count: 11936
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-001.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-002.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-003.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-004.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-005.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-cam0-rgb-006.png
/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb/adl-01-cam0-rgb/adl-01-

In [3]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

data_root = Path('/kaggle/input/datasets/shahliza27/ur-fall-detection-dataset')
png_paths = sorted(data_root.rglob('*.png'))

rows = []
for p in png_paths:
  seq_id = p.parent.name.lower()
  if seq_id.startswith('fall'):
    label = 1
    label_name = 'fall'
  elif seq_id.startswith('adl'):
    label = 0
    label_name = 'adl'
  else:
    continue

  rows.append({
      'path': str(p),
      'seq_id': seq_id,
      'label': label,
      'label_name': label_name,
  })

df = pd.DataFrame(rows)
print('Total PNGs:', len(df))
print('Unique sequences:', df['seq_id'].nunique())
print(df.groupby(['label_name', 'seq_id']).size().head(10))

# 1) Hold out 20% of sequences for test
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_val_idx, test_idx = next(
    gss_test.split(df, df['label'], groups=df['seq_id'])
)
train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# 2) Split remaining sequences into ~85% train / 15% val
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(
    gss_val.split(
        train_val_df, train_val_df['label'], groups=train_val_df['seq_id']
    )
)
train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

assert set(train_df['seq_id']).isdisjoint(val_df['seq_id'])
assert set(train_df['seq_id']).isdisjoint(test_df['seq_id'])
assert set(val_df['seq_id']).isdisjoint(test_df['seq_id'])

train_df['split'] = 'train'
val_df['split'] = 'val'
test_df['split'] = 'test'

manifest = pd.concat([train_df, val_df, test_df], ignore_index=True)
manifest.to_csv('/kaggle/working/ur_fall_manifest.csv', index=False)
print('Saved manifest to /kaggle/working/ur_fall_manifest.csv')
print(
    manifest.groupby('split').agg(
        frames=('path', 'count'), sequences=('seq_id', 'nunique')
    )
)

Total PNGs: 11936
Unique sequences: 70
label_name  seq_id         
adl         adl-01-cam0-rgb    150
            adl-02-cam0-rgb    180
            adl-03-cam0-rgb    180
            adl-04-cam0-rgb    150
            adl-05-cam0-rgb    180
            adl-06-cam0-rgb    230
            adl-07-cam0-rgb    180
            adl-08-cam0-rgb    180
            adl-09-cam0-rgb    150
            adl-10-cam0-rgb    300
dtype: int64
Saved manifest to /kaggle/working/ur_fall_manifest.csv
       frames  sequences
split                   
test     2653         14
train    7926         47
val      1357          9


In [4]:
import pandas as pd

manifest = pd.read_csv("/kaggle/working/ur_fall_manifest.csv")
print(
    pd.crosstab(
        manifest["split"], manifest["label_name"], margins=True, margins_name="Total"
    )
)
print(
    manifest.groupby(["split", "label_name"])["seq_id"]
    .nunique()
    .unstack(fill_value=0)
)

label_name   adl  fall  Total
split                        
test        2286   367   2653
train       5610  2316   7926
val         1045   312   1357
Total       8941  2995  11936
label_name  adl  fall
split                
test         10     4
train        25    22
val           5     4


In [6]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42

manifest = pd.read_csv("/kaggle/working/ur_fall_manifest.csv")
train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

# Compute class weights from train split only to handle ADL vs. Fall imbalance
classes = np.array([0, 1])
weights = compute_class_weight(
    class_weight="balanced", classes=classes, y=train_df["label"].values
)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights (0=adl, 1=fall):", class_weight)


def load_image(path, label):
  img = tf.io.read_file(path)
  img = tf.image.decode_png(img, channels=3)
  img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
  img = tf.cast(img, tf.float32)
  img = preprocess_input(img)
  return img, tf.cast(label, tf.float32)


def make_ds(df, shuffle=False):
  ds = tf.data.Dataset.from_tensor_slices(
      (df["path"].values, df["label"].values)
  )
  if shuffle:
    ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
  ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
  return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_ds(train_df, shuffle=True)
val_ds = make_ds(val_df, shuffle=False)
test_ds = make_ds(test_df, shuffle=False)

I0000 00:00:1795283673.621709      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Class weights (0=adl, 1=fall): {0: 0.7064171122994652, 1: 1.711139896373057}


E0000 00:00:1795283676.148806      50 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
from tensorflow.keras import layers, models

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", mode="max", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath="/kaggle/working/vision_cnn_best.keras",
        monitor="val_auc",
        mode="max",
        save_best_only=True,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    class_weight=class_weight,
    callbacks=callbacks,
)

test_metrics = model.evaluate(test_ds, return_dict=True)
print("Test metrics:", {k: round(float(v), 4) for k, v in test_metrics.items()})
model.save("/kaggle/working/vision_cnn_baseline.keras")

Epoch 1/8
248/248 ━━━━━━━━━━━━━━━━━━━━ 775s 3s/step - accuracy: 0.9802 - auc: 0.9997 - loss: 0.0453 - precision: 0.9386 - recall: 0.9974 - val_accuracy: 0.7701 - val_auc: 0.8173 - val_loss: 1.0056 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/8
248/248 ━━━━━━━━━━━━━━━━━━━━ 787s 3s/step - accuracy: 0.9994 - auc: 1.0000 - loss: 0.0039 - precision: 0.9983 - recall: 0.9996 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 0.0014 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 3/8
248/248 ━━━━━━━━━━━━━━━━━━━━ 787s 3s/step - accuracy: 0.9999 - auc: 1.0000 - loss: 0.0014 - precision: 0.9996 - recall: 1.0000 - val_accuracy: 0.5409 - val_auc: 0.9364 - val_loss: 1.5271 - val_precision: 0.3337 - val_recall: 1.0000
Epoch 4/8
248/248 ━━━━━━━━━━━━━━━━━━━━ 778s 3s/step - accuracy: 0.9995 - auc: 1.0000 - loss: 0.0024 - precision: 0.9983 - recall: 1.0000 - val_accuracy: 1.0000 - val_auc: 1.0000 - val_loss: 7.6437e-04 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 5/8
248/248 ━━